# CAPR benchmark script, run on the free sample

This notebook runs the **unmodified** `../validation/benchmark_script.py` on the 500-patient free sample.

**Read this before the numbers.** The script was written for the full 15,000-patient delivery and reads `../DATASET/capr_wide_patients.parquet` and `../DATASET/capr_longitudinal_rehab.parquet`. The sample files carry a `_sample_` infix, so the notebook copies them under the names the script expects into a temporary folder. Nothing in the script is changed.

With 500 patients, four of the five folds of task A2 hold only a handful of K2 cases, and the fold-to-fold spread is large. The numbers below show that the pipeline runs and what the sample looks like. They are **not** estimates of the full-cohort performance. The published numbers come from the full cohort and are shown beside them.

In [1]:
import os, shutil, subprocess, sys, tempfile
import pandas as pd
pd.set_option('display.width', 200)
SAMPLE = os.path.abspath('../sample')
SCRIPT = os.path.abspath('../validation/benchmark_script.py')
tmp = tempfile.mkdtemp()
os.makedirs(f'{tmp}/VALIDATION'); os.makedirs(f'{tmp}/DATASET')
shutil.copy(SCRIPT, f'{tmp}/VALIDATION/benchmark_script.py')
shutil.copy(f'{SAMPLE}/capr_sample_wide_patients.parquet', f'{tmp}/DATASET/capr_wide_patients.parquet')
shutil.copy(f'{SAMPLE}/capr_sample_longitudinal_rehab.parquet', f'{tmp}/DATASET/capr_longitudinal_rehab.parquet')
out = subprocess.run([sys.executable, 'benchmark_script.py'], cwd=f'{tmp}/VALIDATION', capture_output=True, text=True)
print(out.stdout[-3500:]); print(out.stderr[-500:])

CAPR benchmark · seed 20261006
versions: python 3.13.18, numpy 2.5.3, pandas 3.0.5, sklearn 1.9.1, pyarrow 26.0.1
Task A1: n=431, classes={'K3': 291, 'K4': 91, 'K1': 29, 'K2': 20}
  A1_klevel_4class_functional fold 1: acc=0.770 f1_macro=0.665 bal_acc=0.624
  A1_klevel_4class_functional fold 2: acc=0.872 f1_macro=0.744 bal_acc=0.711
  A1_klevel_4class_functional fold 3: acc=0.733 f1_macro=0.669 bal_acc=0.632
  A1_klevel_4class_functional fold 4: acc=0.826 f1_macro=0.800 bal_acc=0.762
  A1_klevel_4class_functional fold 5: acc=0.849 f1_macro=0.823 bal_acc=0.780
Task A2: n=311, K3 share=0.936
  A2_klevel_K2vsK3_functional fold 1: acc=0.984 auc=0.983 bal_acc=0.875
  A2_klevel_K2vsK3_functional fold 2: acc=0.952 auc=0.983 bal_acc=0.625
  A2_klevel_K2vsK3_functional fold 3: acc=0.968 auc=0.961 bal_acc=0.750
  A2_klevel_K2vsK3_functional fold 4: acc=1.000 auc=1.000 bal_acc=1.000
  A2_klevel_K2vsK3_functional fold 5: acc=1.000 auc=1.000 bal_acc=1.000
Task B: n=500, prevalence=0.1360
  B_return_

In [2]:
sample = pd.read_parquet(f'{tmp}/VALIDATION/benchmark_results.parquet')
full = pd.read_parquet('../validation/benchmark_results.parquet')
cols = ['accuracy', 'balanced_accuracy', 'roc_auc']
a = sample.groupby('task')[cols].mean().add_suffix('_sample')
b = full.groupby('task')[cols].mean().add_suffix('_full_cohort')
s = sample.groupby('task')['roc_auc'].std().rename('roc_auc_sd_across_folds_sample')
print((a.join(b).join(s)).round(3).T)

task                            A1_klevel_4class_functional  A2_klevel_K2vsK3_functional  B_return_to_duty_intake  C_pain_chronicity_early
accuracy_sample                                       0.810                        0.981                    0.898                    0.754
balanced_accuracy_sample                              0.702                        0.850                    0.667                    0.600
roc_auc_sample                                          NaN                        0.985                    0.873                    0.724
accuracy_full_cohort                                  0.822                        0.966                    0.917                    0.778
balanced_accuracy_full_cohort                         0.763                        0.848                    0.725                    0.621
roc_auc_full_cohort                                     NaN                        0.977                    0.899                    0.704
roc_auc_sd_across_folds_sam

## How to read the comparison

- `*_full_cohort` columns are the means of the shipped `benchmark_results.parquet` (15,000 patients) and are the figures that the validation report quotes.
- `*_sample` columns come from this run on 500 patients. The gap, and the spread across folds, come from the small n.
- Task A2 (K2 against K3) looks even better on the sample than on the full cohort. The K-level and the walking tests come from one generated functional continuum, so this task is close to circular by construction, and a small sample only makes it noisier.
- Package versions here are newer than those named in the script header (the shipped results used Python 3.11, pandas 2.3.1, scikit-learn 1.7.1), so the third decimal can move.

In [3]:
# Integrity of the sample: row counts and joins
t = {f[:-8]: pd.read_parquet(f'{SAMPLE}/{f}') for f in sorted(os.listdir(SAMPLE)) if f.endswith('.parquet')}
print({k: v.shape for k, v in t.items()})
pid = set(t['capr_sample_wide_patients']['patient_id'])
print('limbs resolve to patients:', set(t['capr_sample_limbs']['patient_id']) <= pid)
print('prostheses resolve to patients:', set(t['capr_sample_prostheses']['patient_id']) <= pid)
print('pain panel resolves to patients:', set(t['capr_sample_longitudinal_rehab']['patient_id']) <= pid)
print('OMOP persons resolve to patients:', set(t['omop_sample_person']['person_id']) <= pid if 'person_id' in t['omop_sample_person'] else 'person_id column not found')

{'capr_sample_limbs': (690, 20), 'capr_sample_longitudinal_rehab': (2500, 8), 'capr_sample_prostheses': (690, 16), 'capr_sample_wide_patients': (500, 55), 'omop_sample_condition_occurrence': (756, 4), 'omop_sample_device_exposure': (638, 4), 'omop_sample_measurement': (6823, 5), 'omop_sample_observation': (2390, 5), 'omop_sample_person': (500, 8), 'omop_sample_procedure_occurrence': (1688, 4)}
limbs resolve to patients: True
prostheses resolve to patients: True
pain panel resolves to patients: True
OMOP persons resolve to patients: True
